In [ ]:

https://orgservice-prod.setu.co/v1/

In [ ]:
POST /consents
{
  "consentDuration": {
    "unit" : "MONTH",
    "value": "4"
  },
  "vua":  "999999999", // {{mobile}} or {{mobile}}@handle. Ex: 999999999@setu or 999999999@onemoney
  "dataRange": {
      "from": "2020-04-01T00:00:00Z",
      "to": "2023-01-01T00:00:00Z"
  },
  "context": [
  ],
  "additionalParams": {
    "tags": ["Loan_Tracking", "Partner_X"]
  }
}

### Sample: `POST /consents`

Call **Setu orgservice** to create a consent. There is **no separate “org bearer”** env var—auth is whatever **Setu Bridge / API keys** docs say for your product (often a **JWT** or **OAuth**-derived value in `Authorization`, plus headers like product instance id). Build that in your app and pass it via the `headers` argument (or generate the token first, then call this).

Requires: `httpx` (already in this repo’s `requirements.txt`).

In [1]:
from typing import Any, Dict, List, Optional

import httpx

SETU_ORG_BASE_URL = "https://orgservice-prod.setu.co/v1"


def create_setu_consent(
    *,
    vua: str,
    headers: Dict[str, str],
    consent_months: str = "4",
    data_from: str = "2020-04-01T00:00:00Z",
    data_to: str = "2023-01-01T00:00:00Z",
    tags: Optional[List[str]] = None,
    context: Optional[List[Dict[str, Any]]] = None,
    timeout_s: float = 60.0,
) -> httpx.Response:
    """
    POST https://orgservice-prod.setu.co/v1/consents

    vua: mobile number or mobile@handle (e.g. "999999999", "999999999@setu").

    headers: **All auth / product headers Setu requires** for this API (from their docs or
    dashboard—e.g. JWT in `Authorization`, `x-product-instance-id`, etc.). No magic env var.
    """
    url = f"{SETU_ORG_BASE_URL.rstrip('/')}/consents"
    payload: Dict[str, Any] = {
        "consentDuration": {"unit": "MONTH", "value": str(consent_months)},
        "vua": vua,
        "dataRange": {"from": data_from, "to": data_to},
        "context": list(context or []),
        "additionalParams": {"tags": list(tags or ["Loan_Tracking", "Partner_X"])},
    }
    merged = {
        "Content-Type": "application/json",
        "Accept": "application/json",
        **headers,
    }

    with httpx.Client(timeout=timeout_s) as client:
        return client.post(url, json=payload, headers=merged)


# Example: fill `headers` from Setu’s spec for your product (illustrative keys only).
r = create_setu_consent(
    vua="999999999@setu",
    headers={
        "Authorization": "Bearer <jwt_or_oauth_token_from_setu>",
        # "x-product-instance-id": "<from_dashboard>",
    },
)
print(r.status_code, r.text)

403 <html>
<head><title>403 Forbidden</title></head>
<body>
<center><h1>403 Forbidden</h1></center>
</body>
</html>

